# SECTION 1: CNN MODEL TRAINING

In [3]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (15880, 186, 130)
y shape: (15880,)


In [2]:
# ========== IMPORT LIBRARIES ==========
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from copy import deepcopy

# ========== SETUP ==========
SAMPLE_RATE = 22050
BATCH_SIZE = 32
NUM_CLASSES = 5

# ✅ Load matching data files
X = np.load("X_balanced.npy", mmap_mode='r').astype(np.float32)
y = np.load("y_balanced.npy")  # MAKE SURE THIS MATCHES X

print("✅ Data shapes — X:", X.shape, "| y:", y.shape)
assert X.shape[0] == y.shape[0], "❌ Feature-label mismatch"

class_names = ["belly pain", "burping", "discomfort", "hungry", "tired"]

# ========== MODEL DEFINITIONS ==========
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * (X.shape[1] // 4) * (X.shape[2] // 4), 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.fc(self.conv(x))

# ========== DATA LOADER ==========
def get_data_loaders(split_ratio=(0.8, 0.1, 0.1), batch_size=32):
    test_size = split_ratio[2]
    val_size = split_ratio[1] / (split_ratio[0] + split_ratio[1])

    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=test_size, stratify=y, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_size, stratify=y_temp, random_state=42)

    def to_loader(X_part, y_part):
        X_tensor = torch.tensor(X_part, dtype=torch.float32).unsqueeze(1)
        y_tensor = torch.tensor(y_part, dtype=torch.long)
        return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=True)

    return (
        to_loader(X_train, y_train),
        to_loader(X_val, y_val),
        to_loader(X_test, y_test),
        torch.tensor(y_test, dtype=torch.long)
    )

# ========== TRAIN ==========
def train_model(model, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=60, patience=5, save_path="best_model_cnn.pth"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss()

    optimizer = {
        "Adam": torch.optim.Adam,
        "SGD": torch.optim.SGD,
        "RMSprop": torch.optim.RMSprop,
        "AdamW": torch.optim.AdamW
    }.get(optimizer_type, torch.optim.Adam)(model.parameters(), lr=lr)

    best_val_acc = 0
    no_improve_epochs = 0
    results_list = []

    train_accuracies, val_accuracies = [], []
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        # Train
        model.train()
        correct, total, running_loss = 0, 0, 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * yb.size(0)
            correct += (pred.argmax(1) == yb).sum().item()
            total += yb.size(0)

        train_acc = correct / total
        train_loss = running_loss / total
        train_accuracies.append(train_acc)
        train_losses.append(train_loss)

        # Validate
        model.eval()
        val_preds, val_labels, val_loss_sum = [], [], 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                loss = loss_fn(pred, yb)
                val_loss_sum += loss.item() * yb.size(0)
                val_preds.extend(pred.argmax(1).cpu().numpy())
                val_labels.extend(yb.cpu().numpy())

        val_acc = accuracy_score(val_labels, val_preds)
        val_loss = val_loss_sum / len(val_labels)
        val_accuracies.append(val_acc)
        val_losses.append(val_loss)

        print(f"Epoch {epoch+1}/{epochs} - Train Acc: {train_acc*100:.2f}% - Val Acc: {val_acc*100:.2f}% - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience:
                print(f"⏹️ Early stopping at epoch {epoch+1}")
                break

    # Reload best model
    model.load_state_dict(torch.load(save_path))

    # Plot Accuracy
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(train_accuracies)+1), [a*100 for a in train_accuracies], label="Train Accuracy")
    plt.plot(range(1, len(val_accuracies)+1), [a*100 for a in val_accuracies], label="Val Accuracy")
    plt.title("Accuracy over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Plot Loss
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(train_losses)+1), train_losses, label="Train Loss")
    plt.plot(range(1, len(val_losses)+1), val_losses, label="Val Loss")
    plt.title("Loss over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    print(f"\n✅ Final Train Accuracy: {train_accuracies[-1]*100:.2f}%")
    print(f"✅ Best Validation Accuracy: {best_val_acc*100:.2f}%")
    return model, train_accuracies[-1], best_val_acc

# ========== EVALUATE ==========
def evaluate_model(model, test_loader, y_true):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    all_preds = []

    with torch.no_grad():
        for xb, _ in test_loader:
            xb = xb.to(device)
            pred = model(xb)
            all_preds.extend(pred.argmax(1).cpu().numpy())

    acc = accuracy_score(y_true, all_preds)
    print(f"\n✅ Final Test Accuracy: {acc*100:.2f}%")
    print("\n📊 Classification Report:")
    print(classification_report(y_true, all_preds, target_names=class_names))

    cm = confusion_matrix(y_true, all_preds)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(cmap='Blues')
    plt.title("Confusion Matrix")
    plt.grid(False)
    plt.tight_layout()
    plt.show()

    return acc



✅ Data shapes — X: (15880, 186, 130) | y: (15880,)


In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.001 | 60 Epochs | Split 80:10:10")
print("==========================")

train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, final_train_acc, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=60)

test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"📌 Summary:\n  - Train Acc: {final_train_acc*100:.2f}%\n  - Val Acc: {best_val_acc*100:.2f}%\n  - Test Acc: {test_acc*100:.2f}%")



🔬 Running: CNN | Adam | 0.001 | 60 Epochs | Split 80:10:10
Epoch 1/60 - Train Acc: 36.41% - Val Acc: 48.11% - Train Loss: 1.4178 - Val Loss: 1.1769
Epoch 2/60 - Train Acc: 60.18% - Val Acc: 60.33% - Train Loss: 0.9641 - Val Loss: 0.9085
Epoch 3/60 - Train Acc: 71.06% - Val Acc: 66.44% - Train Loss: 0.7080 - Val Loss: 0.8141


In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.001 | 60 Epochs | Split 80:10:10")
print("==========================")

train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, final_train_acc, best_val_acc, _ = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"✅ Final Training Accuracy: {final_train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.001 | 60 Epochs | Split 70:15:15")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.7, 0.15, 0.15))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, final_train_acc, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())


print(f"✅ Final Training Accuracy: {final_train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")


In [ ]:
# E1 experiment: change split, optimizer, or learning rate

print("\n==========================")
print("🔬 Running: CNN | Adam | 0.001 | 60 Epochs | Split 70:20:10")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.7, 0.2, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, final_train_acc, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())


print(f"✅ Final Training Accuracy: {final_train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")


In [ ]:


print("\n==========================")
print("🔬 Running: CNN | SGD | 0.01 | 60 Epochs | Split 80:10:10")
print("==========================")

train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, final_train_acc, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="SGD", lr=0.01, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())


print(f"✅ Final Training Accuracy: {final_train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")


In [ ]:
print("\n==========================")
print("🔬 Running: CNN | RMSprop | 0.001 | 60 Epochs | Split 80:10:10")
print("==========================")

train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, final_train_acc, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="RMSprop", lr=0.001, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())


print(f"✅ Final Training Accuracy: {final_train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")


In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.0005 | 60 Epochs | Split 80:10:10")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, final_train_acc, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.0005, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())


print(f"✅ Final Training Accuracy: {final_train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.0001 | 60 Epochs | Split 80:10:10")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, final_train_acc, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.0001, epochs=60)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())


print(f"✅ Final Training Accuracy: {final_train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.001 | 20 Epochs | Split 80:10:10")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, final_train_acc, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=20)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"✅ Final Training Accuracy: {final_train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
# ========== RUN A SINGLE EXPERIMENT ==========
print("\n==========================")
print("🔬 Running: CNN | Adam | 0.001 | 40 Epochs | Split 80:10:10")
print("==========================")
train_loader, val_loader, test_loader, y_test = get_data_loaders((0.8, 0.1, 0.1))
cnn = CNN(num_classes=NUM_CLASSES)
trained_cnn, final_train_acc, best_val_acc = train_model(cnn, train_loader, val_loader, optimizer_type="Adam", lr=0.001, epochs=40)
test_acc = evaluate_model(trained_cnn, test_loader, y_test.numpy())

print(f"✅ Final Training Accuracy: {final_train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")


In [ ]:

from copy import deepcopy
import os
import pandas as pd
import json

best_val_acc = 0.0
best_model_state = None
best_optimizer_name = ""
results_list = []


In [ ]:

# ================= Final Save Step =================
# Save the best CNN model
best_model_path = f"best_cnn_model_{best_optimizer_name.lower()}.pth"
torch.save(best_model_state, best_model_path)
print(f"✅ Best CNN model saved as {best_model_path} with val acc = {best_val_acc:.4f}")

# Save all results to CSV
df_all_results = pd.DataFrame(results_list)
df_all_results.to_csv("cnn_training_results.csv", index=False)
print("📁 All training results saved to cnn_training_results.csv")

# Save all results to JSON
with open("cnn_training_results.json", "w") as json_file:
    json.dump(results_list, json_file, indent=4)
print("📁 All training results saved to cnn_training_results.json")
